# 05 · 微调与 LoRA：让模型适应你的任务

> GPT-4 通用能力很强，但它不懂你公司的内部术语、不会按你的格式输出、在垂直领域的准确率也未必够用。
> 微调就是把一个通用模型变成专家的过程。

| 你将学到 | 关键词 |
|---------|-------|
| 什么时候需要微调 vs 用 Prompt/RAG | 微调决策树 |
| 全参数微调 vs 参数高效微调（PEFT） | PEFT、冻结层 |
| LoRA 的数学原理 | 低秩分解、W = W₀ + BA |
| 用 PEFT 库给模型注入 LoRA | LoraConfig、get_peft_model |
| 指令微调数据集的格式 | Alpaca、ShareGPT |
| 训练循环与监控 | Trainer、loss 曲线 |


## 第一章：为什么需要微调？

**三种适配 LLM 的方式，适用场景各不同：**

```mermaid
flowchart LR
    NEED(["我需要让 LLM\n完成特定任务"]) --> Q1{"格式/风格\n问题？"}
    Q1 -->|"是"| PROMPT["✅ Prompt Engineering\n零成本，立竿见影"]
    Q1 -->|"需要私有知识"| RAG["✅ RAG\n知识可更新，无需训练"]
    Q1 -->|"模型行为需改变\n大量一致性输出"| FT["✅ 微调\n改变模型本身的能力"]
    FT --> SFT["SFT\n监督微调\n指令遵循"]
    FT --> RLHF["RLHF/DPO\n偏好对齐\n风格调整"]
    FT --> DOMAIN["领域预训练\n专业知识注入"]
```

**微调的典型适用场景：**

| 场景 | 问题 | 微调方案 |
|------|------|---------|
| 医疗报告生成 | 输出格式严格，专业术语多 | SFT + 领域数据 |
| 代码补全 | 特定代码库风格 | SFT on 代码 |
| 客服机器人 | 回复风格、品牌声音 | SFT + DPO |
| 函数调用 | 精确 JSON 格式输出 | SFT on 工具调用数据 |
| 内容安全 | 拒绝特定类别请求 | RLHF/Constitutional AI |

In [ ]:
# 安装依赖（首次运行）
# !pip install transformers peft datasets torch matplotlib

import torch
import math
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional

print(f"PyTorch: {torch.__version__}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

## 第二章：全参数微调 vs 参数高效微调（PEFT）

**全参数微调的问题**：
- 更新所有参数需要存储完整的梯度和优化器状态，显存需求是模型权重的 12-16×
- LLaMA-2 7B 全参微调需要 ~120 GB 显存
- 每个任务保存一份完整模型副本（14 GB × N 个任务）

**PEFT 的思路**：冻结预训练权重，只更新少量新增参数

```mermaid
flowchart LR
    subgraph 全参微调
        A1["预训练权重\n7B 参数"] -->|"全部更新"| A2["微调权重\n7B 参数"]
        A2 --> A3["梯度+优化器\n~50GB"]
    end
    subgraph PEFT_LoRA["LoRA（PEFT 代表方案）"]
        B1["预训练权重\n7B 参数"] -->|"冻结"| B2["冻结权重\n不更新"]
        B3["LoRA 层\n~4M 参数"] -->|"只更新这部分"| B4["梯度+优化器\n~1GB"]
    end
```

**主流 PEFT 方法对比：**

| 方法 | 原理 | 可训练参数比 | 推理开销 |
|------|------|------------|--------|
| LoRA | 低秩矩阵分解 | <1% | 可合并，零开销 |
| Prefix Tuning | 在输入前加软提示 | ~0.1% | 每次推理都有 |
| Adapter | 在层间插入小模块 | ~0.5% | 有轻微延迟 |
| QLoRA | LoRA + 4-bit 量化基础模型 | <1% | 最省显存 |

In [ ]:
# 参数量对比：全参微调 vs LoRA

def model_param_analysis(d_model: int, num_layers: int, rank: int = 8) -> dict:
    """分析 Transformer 的参数量分布"""
    # 每层的 attention 参数：4 个权重矩阵 (Q, K, V, O)，每个 d_model × d_model
    attn_params_per_layer = 4 * d_model * d_model
    # 每层的 FFN 参数：两个线性层，中间维度通常是 4 × d_model
    ffn_params_per_layer = 2 * d_model * (4 * d_model)
    total_params = num_layers * (attn_params_per_layer + ffn_params_per_layer)

    # LoRA 只在 Q, V 矩阵上加（常见配置），每个矩阵加两个低秩矩阵 A(d×r), B(r×d)
    lora_params_per_attn = 2 * (d_model * rank + rank * d_model)  # Q 和 V
    lora_total = num_layers * lora_params_per_attn

    return {
        'total_params': total_params,
        'lora_params': lora_total,
        'lora_ratio': lora_total / total_params,
        'full_ft_memory_gb': total_params * 4 * 16 / 1e9,   # 参数+梯度+Adam×2，FP32
        'lora_ft_memory_gb': lora_total * 4 * 16 / 1e9 + total_params * 2 / 1e9,  # LoRA梯度+冻结FP16
    }

# 典型模型规格
models = {
    'GPT-2 Small': (768, 12),
    'GPT-2 Large': (1280, 36),
    'LLaMA-2 7B': (4096, 32),
    'LLaMA-2 13B': (5120, 40),
}

print(f"{'模型':<15} {'总参数':>10} {'LoRA参数':>10} {'LoRA比例':>8} {'全参显存':>10} {'LoRA显存':>10}")
print('-' * 70)
for name, (d, l) in models.items():
    r = model_param_analysis(d, l, rank=8)
    print(f"{name:<15} {r['total_params']/1e6:>8.0f}M {r['lora_params']/1e6:>8.2f}M "
          f"{r['lora_ratio']:>7.2%} {r['full_ft_memory_gb']:>8.1f}GB {r['lora_ft_memory_gb']:>8.1f}GB")

## 第三章：LoRA 原理深入

**核心洞察**：预训练权重矩阵的更新量（Δ W）本质上是低秩的——任务适应只需要在低维子空间里调整。

**数学表达**：

原来的前向传播：$h = W_0 x$

加入 LoRA 后：$h = W_0 x + \Delta W x = W_0 x + B A x$

其中：
- $W_0 \in \mathbb{R}^{d \times d}$：冻结的预训练权重
- $A \in \mathbb{R}^{r \times d}$：随机高斯初始化（下投影）
- $B \in \mathbb{R}^{d \times r}$：零初始化（上投影），保证训练开始时 $\Delta W = 0$
- $r \ll d$：秩（rank），通常取 4、8、16

**为什么有效**：对于 d=4096, r=8，原矩阵有 16M 参数，LoRA 只有 64K 参数（0.4%），但能捕捉到任务所需的方向变化。

**推理时**：可以把 $W_0 + BA$ 预计算合并，**推理零开销**。

In [ ]:
# 手写 LoRA 层，验证等价性
import torch
import torch.nn as nn

class LoRALinear(nn.Module):
    """在 Linear 层基础上注入 LoRA 旁路"""

    def __init__(self, in_features: int, out_features: int, rank: int = 8, alpha: float = 16.0):
        super().__init__()
        self.rank = rank
        self.scaling = alpha / rank  # 缩放因子，控制 LoRA 的更新幅度

        # 冻结的预训练权重
        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.02, requires_grad=False)

        # LoRA 矩阵：A 随机初始化，B 零初始化
        self.lora_A = nn.Parameter(torch.randn(rank, in_features) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 原始路径（冻结）
        base = x @ self.weight.T
        # LoRA 旁路：x → A（降维）→ B（升维）
        lora = x @ self.lora_A.T @ self.lora_B.T * self.scaling
        return base + lora

    def merge_weights(self) -> nn.Linear:
        """推理时合并权重，消除 LoRA 开销"""
        merged = nn.Linear(self.weight.shape[1], self.weight.shape[0], bias=False)
        delta_w = (self.lora_B @ self.lora_A) * self.scaling
        merged.weight = nn.Parameter(self.weight + delta_w)
        return merged

# 验证：LoRA 输出 = 合并后权重输出
torch.manual_seed(42)
d_in, d_out, r = 64, 64, 8
lora_layer = LoRALinear(d_in, d_out, rank=r)
merged_layer = lora_layer.merge_weights()

x = torch.randn(4, d_in)
out_lora = lora_layer(x)
out_merged = merged_layer(x)

max_diff = (out_lora - out_merged).abs().max().item()
print(f"LoRA 旁路 vs 合并权重的最大误差: {max_diff:.2e}")
print(f"（误差来自浮点精度，可忽略）\n")

# 参数统计
total_params = sum(p.numel() for p in lora_layer.parameters())
trainable_params = sum(p.numel() for p in lora_layer.parameters() if p.requires_grad)
print(f"总参数: {total_params:,}")
print(f"可训练参数（LoRA）: {trainable_params:,}")
print(f"可训练比例: {trainable_params/total_params:.1%}")

## 第四章：用 PEFT 库给 GPT-2 注入 LoRA

PEFT（Parameter-Efficient Fine-Tuning）是 Hugging Face 开发的库，三行代码注入 LoRA，支持 LLaMA、Mistral、GPT-2 等所有主流模型。

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

try:
    from peft import LoraConfig, get_peft_model, TaskType
    PEFT_AVAILABLE = True
except ImportError:
    print("请安装：pip install peft")
    PEFT_AVAILABLE = False

if PEFT_AVAILABLE:
    # 加载基础模型（GPT-2 Small，117M 参数，下载约 500MB）
    model_name = "gpt2"
    print(f"加载模型 {model_name}...")
    model = AutoModelForCausalLM.from_pretrained(model_name)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # 配置 LoRA：只在 attention 的 Q, V 矩阵注入
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=8,                          # LoRA 秩
        lora_alpha=16,               # 缩放因子 α
        target_modules=["c_attn"],   # GPT-2 的 QKV 合并矩阵
        lora_dropout=0.1,
        bias="none",
    )

    # 三行代码注入 LoRA
    peft_model = get_peft_model(model, lora_config)
    peft_model.print_trainable_parameters()

    # 可视化哪些层被冻结
    frozen, trainable = [], []
    for name, param in peft_model.named_parameters():
        if param.requires_grad:
            trainable.append((name, param.numel()))
        else:
            frozen.append((name, param.numel()))

    print(f"\n冻结层数量: {len(frozen)}，可训练层数量: {len(trainable)}")
    print("\n可训练参数（LoRA）：")
    for name, n in trainable[:10]:  # 展示前10个
        print(f"  {name}: {n:,} params")

## 第五章：指令微调数据集格式

微调效果 70% 取决于数据质量。数据格式需要和训练时的模板严格对齐。

**Alpaca 格式**（单轮指令）：
```json
{"instruction": "将下面的句子翻译成英文", "input": "今天天气很好", "output": "The weather is great today."}
```

**ShareGPT 格式**（多轮对话）：
```json
{"conversations": [{"from": "human", "value": "..."}, {"from": "gpt", "value": "..."}]}
```

**训练时的 Prompt 模板**（Alpaca 风格）：
```
Below is an instruction...\n\n### Instruction:\n{instruction}\n\n### Input:\n{input}\n\n### Response:\n{output}
```

**关键原则**：
- 只在 Response 部分计算损失（Instruction 部分 mask 掉）
- 数据量：SFT 通常需要 1K-100K 条高质量样本
- 质量 >> 数量：1000 条精心挑选的数据 > 10万条噪声数据

In [ ]:
# 构造指令微调数据集示例

ALPACA_TEMPLATE = (
    "Below is an instruction that describes a task, paired with an input that provides further context. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n"
    "### Input:\n{input}\n\n"
    "### Response:\n{output}"
)

ALPACA_TEMPLATE_NO_INPUT = (
    "Below is an instruction that describes a task. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n"
    "### Response:\n{output}"
)

sample_data = [
    {
        "instruction": "将下面的句子翻译成英文。",
        "input": "大型语言模型正在改变人工智能领域。",
        "output": "Large language models are transforming the field of artificial intelligence."
    },
    {
        "instruction": "解释什么是注意力机制。",
        "input": "",
        "output": "注意力机制是一种让模型在处理序列时动态关注不同位置重要性的技术..."
    },
    {
        "instruction": "将以下 Python 代码转换为 NumPy 向量化实现。",
        "input": "result = []\nfor x in data:\n    result.append(x * 2 + 1)",
        "output": "result = data * 2 + 1"
    },
]

def format_sample(sample: dict) -> str:
    if sample['input'].strip():
        return ALPACA_TEMPLATE.format(**sample)
    return ALPACA_TEMPLATE_NO_INPUT.format(**sample)

for i, sample in enumerate(sample_data):
    formatted = format_sample(sample)
    print(f"--- 样本 {i+1} ---")
    print(formatted[:300] + ("..." if len(formatted) > 300 else ""))
    print(f"（总长度: {len(formatted)} 字符）\n")

## 第六章：训练循环与监控

Hugging Face `Trainer` 封装了训练循环、梯度累积、混合精度、checkpoint 保存等所有细节。

In [ ]:
# 训练循环演示（小规模模拟，展示关键配置）

from transformers import TrainingArguments

# LoRA 微调的推荐 TrainingArguments
training_args = TrainingArguments(
    output_dir="./lora_output",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,    # 等效 batch_size = 16
    warmup_steps=100,
    learning_rate=3e-4,               # LoRA 通常用比全参更大的学习率
    fp16=torch.cuda.is_available(),   # 混合精度训练
    logging_steps=10,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",                 # 关闭 wandb 等外部日志
)

print("推荐 LoRA 微调配置：")
key_args = ['num_train_epochs', 'per_device_train_batch_size', 'gradient_accumulation_steps',
            'learning_rate', 'fp16', 'warmup_steps']
for k in key_args:
    print(f"  {k}: {getattr(training_args, k)}")

# 模拟 loss 曲线（展示典型的训练趋势）
np.random.seed(42)
steps = np.arange(0, 200)
base_loss = 3.5 * np.exp(-steps / 80) + 1.5
noise = np.random.normal(0, 0.1, len(steps))
train_loss = base_loss + noise
val_loss = base_loss * 1.05 + np.random.normal(0, 0.05, len(steps))

plt.figure(figsize=(9, 4))
plt.plot(steps, train_loss, label='训练 Loss', color='#3498db', alpha=0.7)
plt.plot(steps[::10], val_loss[::10], 'o-', label='验证 Loss', color='#e74c3c', linewidth=2)
plt.axvline(x=67, color='gray', linestyle='--', alpha=0.5, label='Epoch 1')
plt.axvline(x=134, color='gray', linestyle='--', alpha=0.5, label='Epoch 2')
plt.xlabel('训练步数')
plt.ylabel('Loss')
plt.title('LoRA 微调的典型 Loss 曲线')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n训练监控要点：")
print("1. 验证 Loss 持续下降 → 训练正常")
print("2. 训练 Loss 下降但验证 Loss 上升 → 过拟合，减少 epoch 或增大 dropout")
print("3. Loss 不下降 → 学习率过低或数据格式有问题")
print("4. Loss 剧烈震荡 → 学习率过高，尝试降低 10×")

## 第七章：QLoRA——在消费级 GPU 上微调大模型

**QLoRA = INT4 量化基础模型 + LoRA**

Tim Dettmers（bitsandbytes 作者）2023 年提出，使得在单张 24GB RTX 3090 上微调 LLaMA-2 65B 成为可能。

**三个关键技术**：
1. **NF4（Normal Float 4）**：专为正态分布权重设计的 4-bit 数据类型，比 INT4 误差更小
2. **Double Quantization**：量化尺度因子本身，再节省约 0.37 bits/参数
3. **Paged Optimizer**：用 CPU 内存作为 GPU 显存的分页后备，处理显存峰值

```python
# QLoRA 加载方式（需要 bitsandbytes）
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-hf",
    quantization_config=bnb_config,
    device_map="auto",
)
```

| 配置 | 模型 | 显存需求 | 典型 GPU |
|------|------|---------|--------|
| LoRA FP16 | 7B | ~18 GB | A100 |
| QLoRA INT4 | 7B | ~6 GB | RTX 3060 |
| QLoRA INT4 | 13B | ~10 GB | RTX 3080 |
| QLoRA INT4 | 70B | ~40 GB | A100 80G |


## 小结：何时微调？用什么方法？

```mermaid
flowchart TD
    START(["想改善模型效果"]) --> Q1{"问题是什么？"}
    Q1 -->|"缺乏私有知识"| RAG["✅ RAG\n知识可实时更新"]
    Q1 -->|"输出格式/风格不对"| Q2{"改 Prompt 能解决？"}
    Q1 -->|"基础能力不足"| FT
    Q2 -->|"能"| PROMPT["✅ Prompt Engineering\n零成本"]
    Q2 -->|"不能，需要一致性"| FT{"显存预算？"}
    FT -->|"有 A100 80G+"| FULL["全参微调\n最高质量"]
    FT -->|"有 24G 消费卡"| LORA["✅ LoRA\nr=8~32, alpha=16~64"]
    FT -->|"只有 8~16G"| QLORA["✅ QLoRA\nINT4 + LoRA"]
    LORA & QLORA --> MERGE["训练完成后合并权重\n推理零开销"]
```

**LoRA 超参速查**：

| 参数 | 推荐值 | 说明 |
|------|--------|------|
| `r`（秩） | 8~32 | 越大越强，但参数量也越多 |
| `lora_alpha` | 2× r | 通常设为 r 的 2 倍 |
| `target_modules` | Q, V（或 Q, K, V, O） | 至少覆盖 Q 和 V |
| `lora_dropout` | 0.05~0.1 | 防止过拟合 |
| 学习率 | 1e-4 ~ 5e-4 | 比全参大 10× |

**延伸阅读**：
- [LoRA 原论文](https://arxiv.org/abs/2106.09685)（Hu et al., 2021）
- [QLoRA 论文](https://arxiv.org/abs/2305.14314)（Dettmers et al., 2023）
- [PEFT 文档](https://huggingface.co/docs/peft)

**下一步**：[../03_projects/](../03_projects/) — 把微调后的模型部署到真实应用中。